In [7]:
# run twice

import kagglehub

# Download latest version
path = kagglehub.dataset_download("gabrielfcarvalho/cardd-with-yolo-annotations-images-labels")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cardd-with-yolo-annotations-images-labels' dataset.
Path to dataset files: /kaggle/input/cardd-with-yolo-annotations-images-labels


In [8]:
import yaml
from pathlib import Path

def create_yolo_config():
    print("="*60)
    print("GENERATING YOLO DATASET CONFIG")
    print("="*60)

    # The root path as confirmed by the user
    dataset_root = "/kaggle/input/cardd-with-yolo-annotations-images-labels"

    # Define class names exactly as per the dataset documentation provided
    damage_categories = ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']

    # Define the YAML structure
    # We use absolute paths here to ensure the training script finds the data regardless of the working directory
    yaml_data = {
        'path': dataset_root,
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'nc': len(damage_categories),
        'names': damage_categories
    }

    # Save to current working directory so it's accessible for training
    yaml_path = Path("cardd_dataset.yaml")
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_data, f, default_flow_style=False)

    print(f"\n✅ Success! Dataset configuration saved to: {yaml_path.absolute()}")
    print(f"\nSummary of configuration:")
    print(f"- Root: {dataset_root}")
    print(f"- Training images: {dataset_root}/train/images")
    print(f"- Classes: {damage_categories}")

if __name__ == '__main__':
    create_yolo_config()

GENERATING YOLO DATASET CONFIG

✅ Success! Dataset configuration saved to: /content/cardd_dataset.yaml

Summary of configuration:
- Root: /kaggle/input/cardd-with-yolo-annotations-images-labels
- Training images: /kaggle/input/cardd-with-yolo-annotations-images-labels/train/images
- Classes: ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']


In [9]:
!pip install ultralytics

import os
import sys
import shutil
import argparse
import torch
import importlib
from ultralytics import YOLO

# ==============================================================================
# FACTORY RESET PATCH - FORCIBLY BREAKING RECURSION
# ==============================================================================
def apply_factory_reset_patch():
    # 1. Forcibly reload the serialization module to get a CLEAN copy of the code
    import torch.serialization
    importlib.reload(torch.serialization)

    # 2. Get the truly original load function from the fresh module
    original_load = torch.serialization.load

    def safe_load(*args, **kwargs):
        # Force weights_only=False for Ultralytics compatibility
        if 'weights_only' in kwargs:
            kwargs['weights_only'] = False
        # Call the fresh original reference
        return original_load(*args, **kwargs)

    # 3. Apply the patch to all possible entry points
    torch.load = safe_load
    torch.serialization.load = safe_load
    print("✅ Factory reset successful. PyTorch loading is now clean and stabilized.")

# Execute the reset before any model loading
apply_factory_reset_patch()
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

def main():
    print("="*60)
    print("STARTING TRAINING - CARDD DATASET")
    print("="*60)

    # 1. Setup Arguments
    parser = argparse.ArgumentParser()
    parser.add_argument('--data', type=str, default='/content/cardd_dataset.yaml')
    parser.add_argument('--epochs', type=int, default=50)
    parser.add_argument('--batch', type=int, default=16)
    args, unknown = parser.parse_known_args()

    # 2. Check Dataset
    if not os.path.exists(args.data):
        print(f"❌ Dataset config not found at {args.data}")
        return

    # 3. Device Selection
    device = '0' if torch.cuda.is_available() else 'cpu'
    print(f"Using Device: {device.upper()}")
    if device == 'cpu':
        print("⚠️ WARNING: Training on CPU will be slow. Use GPU for better performance.")

    # 4. Load Pretrained Model
    print("📦 Loading YOLOv8n weights...")
    try:
        model = YOLO('yolov8n.pt')
        print("✅ Model loaded successfully.")
    except Exception as e:
        print(f"❌ Failed to load model: {e}")
        return

    # 5. Training Configuration
    train_args = {
        'data': args.data,
        'epochs': args.epochs,
        'batch': args.batch,
        'imgsz': 640,
        'device': device,
        'optimizer': 'AdamW',
        'project': 'runs/train',
        'name': 'damage_detection_v1',
        'exist_ok': True,
        'save': True,
        'plots': True
    }

    print(f"\n🚀 Starting training sequence...")
    try:
        model.train(**train_args)
        best_path = 'runs/train/damage_detection_v1/weights/best.pt'
        if os.path.exists(best_path):
            shutil.copy(best_path, 'best_damage_model.pt')
            print(f"\n✅ Success! Model saved as 'best_damage_model.pt'")
    except Exception as e:
        print(f"\n❌ Training interrupted: {e}")

if __name__ == '__main__':
    main()

✅ Factory reset successful. PyTorch loading is now clean and stabilized.
STARTING TRAINING - CARDD DATASET
Using Device: 0
📦 Loading YOLOv8n weights...
✅ Model loaded successfully.

🚀 Starting training sequence...
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cardd_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=30

In [15]:
import os, json, cv2, numpy as np
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO
import argparse

def load_model(model_path='/content/runs/detect/runs/train/damage_detection_v1/weights/best.pt'): # Changed default model path
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"❌ Model not found: {model_path}")
    model = YOLO(model_path)
    classes = model.names  # ✅ Dynamically read exact trained classes
    return model, classes

def calculate_severity(conf, bbox, img_shape):
    area = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
    total = img_shape[0] * img_shape[1]
    score = min(conf * 0.7 + (area / total) * 0.3, 1.0)
    if score < 0.3: return 'low', float(score) # Convert to Python float
    if score < 0.7: return 'medium', float(score) # Convert to Python float
    return 'high', float(score) # Convert to Python float

def calculate_location(bbox, img_shape):
    h, w = img_shape[:2]
    cx, cy = (bbox[0] + bbox[2])/2, (bbox[1] + bbox[3])/2

    h_pos = 'left' if cx < w/3 else ('right' if cx > 2*w/3 else 'center')
    v_pos = 'top' if cy < h/3 else ('bottom' if cy > 2*h/3 else 'middle')
    return f"{v_pos}-{h_pos}"

def analyze_images(model, classes, image_paths, conf=0.25):
    print(f"\n📷 Processing {len(image_paths)} images...")
    all_results = []

    # ✅ NATIVE BATCH INFERENCE
    results = model.predict(source=image_paths, conf=conf, device='0' if torch.cuda.is_available() else 'cpu', verbose=False)

    for img_path, res in zip(image_paths, results):
        detections = []
        if res.boxes is not None:
            for box in res.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                conf_val = float(box.conf[0])
                cls_id = int(box.cls[0])
                cls_name = classes[cls_id] if cls_id < len(classes) else 'unknown'

                img_shape = res.orig_shape
                severity, sev_score = calculate_severity(conf_val, [x1,y1,x2,y2], img_shape)
                location = calculate_location([x1,y1,x2,y2], img_shape)

                detections.append({
                    'class_name': cls_name, 'confidence': conf_val,
                    'severity': severity, 'severity_score': sev_score,
                    'location': location, 'bbox': [float(x1), float(y1), float(x2), float(y2)]
                })
        all_results.append({'image': os.path.basename(img_path), 'detections': detections})
    return all_results

def main():
    parser = argparse.ArgumentParser()
    # Removed required=True and added a default path for --images
    parser.add_argument('--images', type=str, default='/kaggle/input/cardd-with-yolo-annotations-images-labels/test/images', help='Path to image(s) or directory')
    parser.add_argument('--model', type=str, default='/content/runs/detect/runs/train/damage_detection_v1/weights/best.pt') # Changed default model path
    parser.add_argument('--conf', type=float, default=0.25)
    parser.add_argument('--output', type=str, default='analysis_results')
    args = parser.parse_args([]) # Modified to parse an empty list of arguments

    model, classes = load_model(args.model)
    print(f"✅ Model loaded. Classes: {list(classes.values())}")

    # Collect image paths
    img_paths = []
    p = Path(args.images)
    if p.is_file():
        img_paths = [str(p)]
    elif p.is_dir():
        exts = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
        for ext in exts:
            img_paths.extend([str(f) for f in p.glob(ext)])

    if not img_paths:
        print("❌ No valid images found.")
        return

    results = analyze_images(model, classes, img_paths, args.conf)

    # Save JSON report
    os.makedirs(args.output, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    report = {'timestamp': ts, 'results': results}
    out_file = os.path.join(args.output, f'report_{ts}.json')
    with open(out_file, 'w') as f:
        json.dump(report, f, indent=2)

    # Save visualizations
    viz_dir = os.path.join(args.output, 'visualizations')
    os.makedirs(viz_dir, exist_ok=True)
    color_map = {'low': (0,255,0), 'medium': (0,255,255), 'high': (0,0,255)}

    for img_path, res in zip(img_paths, results):
        img = cv2.imread(img_path)
        for det in res['detections']:
            x1,y1,x2,y2 = map(int, det['bbox'])
            clr = color_map[det['severity']]
            cv2.rectangle(img, (x1,y1), (x2,y2), clr, 2)
            label = f"{det['class_name']} {det['confidence']:.2f} ({det['severity']})"
            cv2.putText(img, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, clr, 2)
        out_path = os.path.join(viz_dir, f"viz_{os.path.basename(img_path)}")
        cv2.imwrite(out_path, img)

    print(f"\n✅ Analysis complete!")
    print(f"📄 Report: {out_file}")
    print(f"🖼️ Visualizations: {viz_dir}")

if __name__ == "__main__":
    import torch
    main()

✅ Model loaded. Classes: ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']

📷 Processing 374 images...

✅ Analysis complete!
📄 Report: analysis_results/report_20260403_174325.json
🖼️ Visualizations: analysis_results/visualizations
